# E-commerce Orders Dataset — Exploratory Data Analysis (EDA)

This notebook explores the `ecommerce_orders_dataset.csv` file: data overview, cleaning checks,
univariate & bivariate analysis, correlations, and key business insights (sales, customers,
products, geography, returns, profitability).

**Sections**
1. Setup & Load Data
2. Data Overview & Quality Checks
3. Univariate Analysis (Numeric)
4. Univariate Analysis (Categorical)
5. Time-based Trends
6. Bivariate / Relationship Analysis
7. Correlation Heatmap
8. Returns & Order Status Analysis
9. Key Business Insights Summary


## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 50)

# Update this path if needed
DATA_PATH = 'ecommerce_orders_dataset.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['Order_Date'])
df.head()


## 2. Data Overview & Quality Checks

In [ ]:
print('Shape:', df.shape)
df.info()


In [ ]:
# Missing values
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)


In [ ]:
# Duplicate rows
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate Order_IDs:', df['Order_ID'].duplicated().sum())


In [ ]:
# Descriptive statistics for numeric columns
df.describe().T


In [ ]:
# Descriptive statistics for categorical columns
df.describe(include='object').T


## 3. Univariate Analysis — Numeric Columns

In [ ]:
numeric_cols = ['Order_Amount', 'Unit_Price', 'Quantity', 'Discount_Percent',
                'Shipping_Cost', 'Profit_Amount', 'Profit_Margin_Percent',
                'Customer_Age', 'Delivery_Days', 'Review_Rating', 'Customer_Lifetime_Value']

fig, axes = plt.subplots(4, 3, figsize=(16, 16))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(col)
for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots to spot outliers
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
box_cols = ['Order_Amount', 'Unit_Price', 'Shipping_Cost', 'Profit_Amount', 'Customer_Lifetime_Value', 'Delivery_Days']
for i, col in enumerate(box_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(col)
plt.tight_layout()
plt.show()


## 4. Univariate Analysis — Categorical Columns

In [ ]:
cat_cols = ['Product_Category', 'Country', 'Payment_Method', 'Device_Type',
            'Customer_Segment', 'Order_Status', 'Traffic_Source', 'Season']

fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    order = df[col].value_counts().index
    sns.countplot(y=df[col], order=order, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Order Count by {col}')
plt.tight_layout()
plt.show()


## 5. Time-based Trends

In [ ]:
monthly = df.groupby(df['Order_Date'].dt.to_period('M')).agg(
    Orders=('Order_ID', 'count'),
    Revenue=('Order_Amount', 'sum')
).reset_index()
monthly['Order_Date'] = monthly['Order_Date'].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.plot(monthly['Order_Date'], monthly['Orders'], color='steelblue', marker='o', label='Orders')
ax2.plot(monthly['Order_Date'], monthly['Revenue'], color='orange', marker='s', label='Revenue')
ax1.set_ylabel('Orders', color='steelblue')
ax2.set_ylabel('Revenue', color='orange')
ax1.set_xticklabels(monthly['Order_Date'], rotation=90)
plt.title('Monthly Orders & Revenue Trend')
plt.tight_layout()
plt.show()


In [ ]:
# Orders by day of week and season
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
sns.countplot(x='Day_Of_Week', data=df, order=dow_order, ax=axes[0], palette='mako')
axes[0].set_title('Orders by Day of Week')
axes[0].tick_params(axis='x', rotation=45)

sns.countplot(x='Season', data=df, ax=axes[1], palette='crest')
axes[1].set_title('Orders by Season')
plt.tight_layout()
plt.show()


## 6. Bivariate / Relationship Analysis

In [ ]:
# Revenue by category
cat_rev = df.groupby('Product_Category')['Order_Amount'].sum().sort_values(ascending=False)
plt.figure(figsize=(10,5))
sns.barplot(x=cat_rev.values, y=cat_rev.index, palette='flare')
plt.title('Total Revenue by Product Category')
plt.xlabel('Revenue')
plt.show()


In [ ]:
# Average order amount by customer segment
plt.figure(figsize=(8,5))
sns.barplot(x='Customer_Segment', y='Order_Amount', data=df, estimator=np.mean, palette='pastel')
plt.title('Average Order Amount by Customer Segment')
plt.show()


In [ ]:
# Order amount vs customer age (scatter with trend)
plt.figure(figsize=(8,5))
sns.scatterplot(x='Customer_Age', y='Order_Amount', hue='Customer_Segment', data=df.sample(2000, random_state=1), alpha=0.5)
plt.title('Order Amount vs Customer Age (sample)')
plt.show()


In [ ]:
# Payment method vs average delivery days
plt.figure(figsize=(9,5))
sns.barplot(x='Payment_Method', y='Delivery_Days', data=df, estimator=np.mean, palette='cool')
plt.title('Average Delivery Days by Payment Method')
plt.xticks(rotation=30)
plt.show()


## 7. Correlation Heatmap

In [ ]:
corr_cols = ['Unit_Price', 'Quantity', 'Discount_Percent', 'Discount_Amount',
             'Shipping_Cost', 'Tax_Amount', 'Order_Amount', 'Delivery_Days',
             'Review_Rating', 'Customer_Lifetime_Value', 'Profit_Margin_Percent', 'Profit_Amount']

plt.figure(figsize=(11, 9))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()


## 8. Returns & Order Status Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.countplot(x='Order_Status', data=df, order=df['Order_Status'].value_counts().index, ax=axes[0], palette='rocket')
axes[0].set_title('Order Status Distribution')

return_rate_by_cat = df.groupby('Product_Category')['Returned'].apply(lambda x: (x=='Yes').mean()*100).sort_values(ascending=False)
sns.barplot(x=return_rate_by_cat.values, y=return_rate_by_cat.index, ax=axes[1], palette='magma')
axes[1].set_title('Return Rate % by Product Category')
axes[1].set_xlabel('Return Rate (%)')
plt.tight_layout()
plt.show()


## 9. Key Business Insights Summary

Run the cell below to auto-generate a quick text summary of headline metrics.


In [ ]:
total_revenue = df['Order_Amount'].sum()
total_orders = df['Order_ID'].nunique()
avg_order_value = df['Order_Amount'].mean()
total_profit = df['Profit_Amount'].sum()
return_rate = (df['Returned']=='Yes').mean()*100
top_category = df.groupby('Product_Category')['Order_Amount'].sum().idxmax()
top_country = df.groupby('Country')['Order_Amount'].sum().idxmax()

print(f"Total Revenue: {total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Average Order Value: {avg_order_value:,.2f}")
print(f"Total Profit: {total_profit:,.2f}")
print(f"Return Rate: {return_rate:.2f}%")
print(f"Top Product Category (by revenue): {top_category}")
print(f"Top Country (by revenue): {top_country}")


---
### Next step
Use the accompanying **`app.py`** Streamlit dashboard (in the same folder) to explore this
dataset interactively with filters, KPI cards, and charts — just run:

```bash
pip install streamlit plotly pandas
streamlit run app.py
```
